<a href="https://colab.research.google.com/github/mrpunith-1920/gnn-fraud-detection/blob/main/03_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.3 MB/s  0:00:00


In [4]:
import torch
path = '/content/drive/MyDrive/gnn-project/data/elliptic_bitcoin_dataset/'
data = torch.load(path + 'elliptic_graph.pt', weights_only=False)
print(data)


Data(x=[203769, 165], edge_index=[2, 234355], y=[203769], timestep=[203769], train_mask=[203769], test_mask=[203769])


In [5]:
X_train = data.x[data.train_mask].numpy()
y_train = data.y[data.train_mask].numpy()
X_test  = data.x[data.test_mask].numpy()
y_test  = data.y[data.test_mask].numpy()

print(X_train.shape, X_test.shape)    # (29894, 165) (16670, 165)

(29894, 165) (16670, 165)


In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, roc_auc_score

results = []

def evaluate(name, model):
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    row = {
        'Model': name,
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'PR-AUC': average_precision_score(y_test, prob),
        'ROC-AUC': roc_auc_score(y_test, prob),
    }
    results.append(row)
    print(row)

In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                            random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate('Random Forest', rf)

{'Model': 'Random Forest', 'Precision': 0.9893048128342246, 'Recall': 0.6832871652816251, 'F1': 0.8083014746040416, 'PR-AUC': np.float64(0.7937395518347021), 'ROC-AUC': np.float64(0.931243339665409)}


In [9]:
from xgboost import XGBClassifier

ratio = (y_train == 0).sum() / (y_train == 1).sum()   # about 7.6
xgb = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                    scale_pos_weight=ratio, random_state=42, eval_metric='logloss')
xgb.fit(X_train, y_train)
evaluate('XGBoost', xgb)

{'Model': 'XGBoost', 'Precision': 0.8668122270742358, 'Recall': 0.7331486611265005, 'F1': 0.7943971985992997, 'PR-AUC': np.float64(0.7998508971724004), 'ROC-AUC': np.float64(0.9256766935488121)}


In [10]:
import pandas as pd
results_df = pd.DataFrame(results).round(3)
results_df

,Model,Precision,Recall,F1,PR-AUC,ROC-AUC
0,Random Forest,0.989,0.683,0.808,0.794,0.931
1,XGBoost,0.867,0.733,0.794,0.800,0.926


Model	Precision	Recall	What it means
Random Forest	0.989	0.683	When it says “fraud” it is almost always right (99%), but it misses about 32% of the real fraud
XGBoost	0.867	0.733	It catches more fraud (73%) but raises more false alarms (13% of its flags are wrong)
Precision answers: of the transactions flagged as fraud, how many really were?
Recall answers: of all the real fraud, how many did we catch?
F1 is the balance of the two. Random Forest wins slightly on F1 and XGBoost slightly on PR-AUC, so there is no clear winner. Say exactly that in your report.
PR-AUC is the most reliable single number for imbalanced data. You will compare the GNNs against about 0.80.

In a real bank, the choice depends on cost. Missing fraud loses money (you’d favor recall), while false alarms annoy customers (you’d favor precision). This is a good point for your viva.